In [0]:
import pyspark.sql.functions as F

catalog_name = "f1_dev"
target_table = f"{catalog_name}.silver.fact_pit_stops"



In [0]:
# 1. Read raw pit_stops table
df_pit_stops_raw = spark.table(f"{catalog_name}.bronze.pit_stops")



In [0]:
# 2. Transform & Cleanse
df_pit_stops_silver = (
    df_pit_stops_raw
    .replace(r"\N", None)
    .select(
        F.col("raceId").cast("integer").alias("race_id"),
        F.col("driverId").cast("integer").alias("driver_id"),
        F.col("stop").cast("integer").alias("stop_number"),
        F.col("lap").cast("integer").alias("lap_number"),
        F.col("time").alias("pit_stop_time_of_day"),
        F.col("duration").alias("duration_formatted"),
        F.col("milliseconds").cast("long").alias("duration_ms")
    )
)



In [0]:
# 3. Write as Delta table in Silver schema
df_pit_stops_silver.write.mode("overwrite").saveAsTable(target_table)



In [0]:
# 4. Set Unity Catalog Governance Tags
spark.sql(f"""
    ALTER TABLE {target_table} SET TAGS (
        'layer' = 'silver',
        'table_type' = 'fact',
        'contains_pii' = 'false'
    )
""")

print(f"✓ Successfully built and governed: {target_table}")